In [19]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages

class FormsState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    qa_answer: str
    user_input: str
    # filled_pdf_path: str

state: FormsState = {
    "messages":  [HumanMessage(content="điền cho tôi mẫu CC01")],
    "qa_answer": "Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.amazonaws.com/MauCC02.pdf",
    "user_input": "điền cho tôi mẫu CC01",
}

In [20]:
from langchain_core.tools import tool

@tool
def select_forms(qa_answer: str, user_input: int):
    """
    Phân tích câu trả lời từ QA node.
    """

    if user_input > 1:
        return {
            "messages" : f"Đã phân tích nhiều câu trả lời {qa_answer}",
            "value": 2
        }
    else:
        return {
            "messages" : f"Đã phân tích một câu trả lời {qa_answer}",
            "value": 1
        }
        

@tool
def result(phantich: dict):
    """
    Đưa ra kết luận từ  kết quả phân tích

    Args:
        phantich (dict): _description_
    """
    if phantich["value"] == 2:
        return phantich["messages"]
    else:
        return f"Câu trả lời duy nhất {phantich["messages"]}"

    

In [ ]:
from langchain_groq import ChatGroq
# from dotenv import load_dotenv
from langchain.messages import HumanMessage, SystemMessage, AnyMessage
# import asyncio
# import os
GROQ_API_KEY = ""
# load_dotenv()
# GROQ_API_KEY = os.getenv("GROQ_API_KEY")
# # GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

_llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=5,
)
TOOLS = [select_forms, result]
llm_with_tool = _llm.bind_tools(TOOLS)


In [25]:
SYSTEM = "Bạn là agent phân tích. Dùng tool để phân tích và đưa ra kết quả"

user = {
    "qa_answer": "Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.amazonaws.com/MauCC02.pdf",
    "user_input": 3,
}

user_content = f"""
qa_answer: {user["qa_answer"]}
user_input: {user["user_input"]}
"""

messages = [SystemMessage(content=SYSTEM)]
messages.append(HumanMessage(content=user_content))

TOOL_MAP = {"select_forms": select_forms, "result": result}

# Lặp cho đến khi LLM không gọi tool nữa
while True:
    response = llm_with_tool.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        print("\n✅ Kết quả cuối:")
        print(response.content)
        break

    for i, tc in enumerate(response.tool_calls):
        print(f"🔧 Bước {i+1}: Gọi tool [{tc['name']}] với args: {tc['args']}")
        
        tool_fn = TOOL_MAP[tc["name"]]
        tool_result = tool_fn.invoke(tc["args"])
        
        print(f"   ↳ Kết quả: {tool_result}")
        
        messages.append({
            "role": "tool",
            "tool_call_id": tc["id"],
            "content": str(tool_result)
        })

🔧 Bước 1: Gọi tool [select_forms] với args: {'qa_answer': 'Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.amazonaws.com/MauCC02.pdf', 'user_input': 3}
   ↳ Kết quả: {'messages': 'Đã phân tích nhiều câu trả lời Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.amazonaws.com/MauCC02.pdf', 'value': 2}
🔧 Bước 1: Gọi tool [result] với args: {'phantich': {'messages': 'Đã phân tích nhiều câu trả lời Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.amazonaws.com/MauCC02.pdf', 'value': 2}}
   ↳ Kết quả: Đã phân tích nhiều câu trả lời Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.amazonaws.com/MauCC02.pdf

✅ Kết quả cuối:
Đã phân tích nhiều câu trả lời: Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.

In [23]:
# messages.append(response)

# # Thực thi tool calls
# for tc in response.tool_calls:
#     tool_fn = {"select_forms": select_forms, "result": result}[tc["name"]]
#     tool_result = tool_fn.invoke(tc["args"])
#     messages.append({"role": "tool", "tool_call_id": tc["id"], "content": str(tool_result)})

# # Gọi LLM lần 2 để tổng hợp kết quả
# final = llm_with_tool.invoke(messages)
# print(final.content)